In [1]:

!pip install -q python-dotenv 

from dotenv import load_dotenv
load_dotenv('.env') # path relative to notebook

True

In [5]:
"""single_call_wrapper.py

A compact helper that imports the core functions from the existing
`run_hf_bird_model_chatgpt` module and performs **one** OpenAI request for a
list of images.

The file is intentionally short so you can copy the `single_call_main`
function into a notebook later and still import the same helpers.
"""

from pathlib import Path
import json, re

# 1️⃣ Make the `code` directory importable
import sys, os
# Absolute path to the folder that contains the module
dir = os.path.abspath('code')
if dir not in sys.path: sys.path.append(dir)
dir = os.path.abspath('lib')
if dir not in sys.path: sys.path.append(dir)

# 2️⃣ Import the helpers
from run_hf_bird_model_chatgpt import read_image_base64, _openai_chat_completion

# Import the low‑level helpers defined in the main module
# (they are public functions in that script)
#from code.run_hf_bird_model_chatgpt import read_image_base64, _openai_chat_completion


In [ ]:
# ------------------------------------------------------------
# OpenAI GPT‑4o Vision query.
# ------------------------------------------------------------

def my_predict_with_gpt4o(image_path: Path, model_name: str, conf_threshold: float, no_bird_conf: float):
    """Returns (label, label_cn, confidence, raw_json) from GPT‑4o.
    The model is asked to return a JSON object with keys:
    - `label` – English name (lower‑case, or 'unknown')
    - `label_cn` – Chinese name (or placeholder) 
    - `confidence` – float 0‑1
    """
    img_b64 = read_image_base64(image_path)
    system_prompt = (
        "You are an expert bird‑identification system. "
        "For the given image, output a JSON object with three fields: `label` (English, lower‑case), `label_cn` (Chinese name), and `confidence` (float 0‑1)."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + img_b64}}]}
    ]
    try:
        # Debug: show the message payload sent to OpenAI (first 2 lines only)
        response = _openai_chat_completion(messages, model_name)
        content = response['choices'][0]['message']['content']
        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            # Extract JSON block if there is surrounding text
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if not match:
                raise ValueError('No JSON found in OpenAI response')
            data = json.loads(match.group())
        raw_json = json.dumps(data, ensure_ascii=False)
        label = data.get('label', 'unknown').lower()
        label_cn = data.get('label_cn', '未知')
        confidence = float(data.get('confidence', 0.0))
        return label, label_cn, confidence, raw_json
    except Exception as e:
        print(f"⚠️  OpenAI request failed for {image_path.name}: {e}")
        return "unknown", "未知", 0.0, "{}"


In [ ]:


DATA_DIR = "./data"
OUTPUT_DIR = "./output"
MODEL = "gpt-4o"
JPG_DIR = "./data/jpg"
jpg_dir = Path(JPG_DIR)

IMAGE_FILENAME = "_D5D8111.jpg"

image_path = jpg_dir / IMAGE_FILENAME
print(f"Processing image: {image_path.absolute()}")


In [ ]:
#bird_label, bird_label_cn, confidence, raw_json = predict_with_gpt4o(image_path, MODEL, conf_threshold=0.5, no_bird_conf=0.2)
img_b64 = read_image_base64(image_path)
system_prompt = (
    "You are an expert bird‑identification system. "
    "For the given image, output a JSON object with three fields: `label` (English, lower‑case), `label_cn` (Chinese name), and `confidence` (float 0‑1)."
)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + img_b64}}]}
]

import os
print(os.getenv('OPENAI_API_KEY'))


In [ ]:
from run_hf_bird_model_chatgpt import predict_with_gpt4o
label,label_cn,confidence,raw_json = predict_with_gpt4o(image_path, MODEL, conf_threshold=0.8, no_bird_conf=0.2)
print(f"Label: {label}, Label CN: {label_cn}, Confidence: {confidence}")